# Importing

In [ ]:
! source /home/ivm/envs/valid_env/bin/activate
import polars as pl
import numpy as np
import sys
sys.path.append(("/home/ivm/valid/scripts/utils/"))
from general_utils import *

# For shap
from minor_plot_utils import get_plot_names
from labeling_utils import add_ages
import pickle
import shap

from datetime import datetime

%load_ext autoreload
%autoreload 2

# Settings

In [ ]:
base_path="/home/ivm/valid/data/processed_data/kanta_R14"
%env base_path=$base_path
%env fg_ver=R13
%env lab_name=hba1c
%env extra=d0_strong_ld
%env lab_name_two=fgluc
%env extra_two=d0_ld
%env date_two=2026-03-24
%env extra_labels=testv10_2022_w3
%env date_1=2026-04-20
%env date_2=2026-04-24
%env date_3=2026-04-24
%env date_4=2026-04-24
%env date_5=2026-04-22

%env base_date=2022-10-01
base_date=datetime(2022,10,1)
%env val_base_date=2023-10-01
val_base_date=datetime(2023,10,1)
%env final_base_date=2025-12-31
final_base_date=datetime(2025,12,31)

%env start_pred_date=2023-01-01
%env end_pred_date=2023-12-31
%env val_start_pred_date=2024-01-01
%env val_end_pred_date=2024-12-31

%env min_age=30
%env max_age=70
%env months_buffer=3
%env abnorm_type=strong
%env future_val=1
%env n_test=5000
%env version=v10
%env bin_goal=y_MEAN_ABNORM
%env cont_goal=y_MEAN

goal="y_MEAN_ABNORM"
%env extra_labels=v10_2023val2024_w3
file_descr = "v10_2023val2024_w3"
val_name = "val_2024"

%env val_name=val_2024
lab_name = "hba1c"

# Filters

In [ ]:
data = pl.read_parquet(base_path+"/step1_clean/hba1c_d0_strong_ld_2026-04-20.parquet")

train_goals = {"mglogloss": "y_MEAN_ABNORM", "logloss": "y_MEAN_ABNORM",  "q90": "y_MEAN", "q95":"y_MEAN", "q75": "y_MEAN", "q25": "y_MEAN", "q10": "y_MEAN", "mae": "y_MEAN"}

lastval_long_filter = (pl.col.FINNGENID.is_in(data.filter(pl.col.DATE<base_date).filter((pl.col.DATE==pl.col.DATE.max()).over("FINNGENID")).filter(pl.col.DATE.dt.year()<2021)["FINNGENID"].implode()))
no_history_filter = (~pl.col.FINNGENID.is_in(data.filter(pl.col.DATE<base_date)["FINNGENID"].implode()))
history_filter = (pl.col.FINNGENID.is_in(data.filter(pl.col.DATE<base_date)["FINNGENID"].implode()))
last_norm_filter = (pl.col.FINNGENID.is_in(data.filter(pl.col.DATE<base_date).filter((pl.col.DATE==pl.col.DATE.max()).over("FINNGENID")).filter(pl.col.ABNORM_CUSTOM<1)["FINNGENID"].implode()))
no_abnorm_filter = (~pl.col.FINNGENID.is_in(data.filter(pl.col.DATE<base_date).filter(pl.col.VALUE<42)["FINNGENID"].implode()))
thirty_filter = (((pl.col.EVENT_AGE>=30)&(pl.col.EVENT_AGE<40)))
fourty_filter = (((pl.col.EVENT_AGE>=40)&(pl.col.EVENT_AGE<50)))
fifty_filter = (((pl.col.EVENT_AGE>=50)&(pl.col.EVENT_AGE<60)))
sixty_filter = (((pl.col.EVENT_AGE>=60)&(pl.col.EVENT_AGE<=70)))

set_names = {0: "Train", 1: "Valid", 2: "Test"}
test_filter = pl.col.SET==2
valid_filer = pl.col.SET==1
train_filter = pl.col.SET==0

# high variance filter
val_vars = data.filter(pl.col.DATE<base_date).group_by("FINNGENID").agg(pl.col.VALUE.var().alias("VAR"))
low_var_cut = np.percentile(val_vars["VAR"], 33)
high_var_cut = np.percentile(val_vars["VAR"], 67)
high_var_filter = (~pl.col.FINNGENID.is_in(val_vars.filter(pl.col.VAR>=high_var_cut)["FINNGENID"].implode()))
low_var_filter = (~pl.col.FINNGENID.is_in(val_vars.filter(pl.col.VAR<=low_var_cut)["FINNGENID"].implode()))
mid_var_filter = (~pl.col.FINNGENID.is_in(val_vars.filter((pl.col.VAR>=high_var_cut)&(pl.col.VAR<=low_var_cut))["FINNGENID"].implode()))

# low number filter
val_counts = data.filter(pl.col.DATE<base_date).group_by("FINNGENID").agg(pl.col.VALUE.len().alias("N"))
val_counts
low_n_cut = np.percentile(val_counts["N"], 33)
high_n_cut = np.percentile(val_counts["N"], 67)
high_n_filter = (~pl.col.FINNGENID.is_in(val_counts.filter(pl.col.N>high_n_cut)["FINNGENID"].implode()))
low_n_filter = (~pl.col.FINNGENID.is_in(val_counts.filter(pl.col.N<low_n_cut)["FINNGENID"].implode()))
mid_n_filter = (~pl.col.FINNGENID.is_in(val_counts.filter((pl.col.N>=low_n_cut)&(pl.col.N<=high_n_cut))["FINNGENID"].implode()))


thirty_filter = (((pl.col.EVENT_AGE>=30)&(pl.col.EVENT_AGE<40)))
fourty_filter = (((pl.col.EVENT_AGE>=40)&(pl.col.EVENT_AGE<50)))
fifty_filter = (((pl.col.EVENT_AGE>=50)&(pl.col.EVENT_AGE<60)))
sixty_filter = (((pl.col.EVENT_AGE>=60)&(pl.col.EVENT_AGE<=70)))

set_names = {0: "Train", 1: "Valid", 2: "Test"}
test_filter = pl.col.SET==2
valid_filer = pl.col.SET==1
train_filter = pl.col.SET==0

filters = {"All": True, 
           "History": history_filter, 
           "All normal | No history": no_abnorm_filter|no_history_filter, 
           "Last <2021 | No history": lastval_long_filter|no_history_filter, 
           "Last <2021": lastval_long_filter, 
           "Low var": low_var_filter, 
           "Mid var": mid_var_filter, 
           "High var": high_var_filter, 
           "Low N": low_n_filter, 
           "Mid N": mid_n_filter, 
           "High N": high_n_filter, 

           "No history": no_history_filter, 
           "30-40": thirty_filter, 
           "40-50": fourty_filter, 
           "50-60": fifty_filter, 
           "60-70": sixty_filter}

preds_descrs={"1_lab_manualquants_month": "lab sequence", "1_clinpheno": "clinical phenotype", "2_lastval": "last value", "2_sumstats": "sumstats", "3_twosumstats": "eGFR+Cystatin C", "3_twosumstats_2": "eGFR+UACR", "3_otherlabs": "other labs", "3_registry": "registry data", "4_all": "all data", "5_icd": "ICD data", "5_atc": "ATC data"}
metrics = ["logloss", "q10", "q25", "mae"]

goal_names = {"y_MEAN_ABNORM": "Mean abnormal", "y_NEXT_ABNORM": "Next abnormal"}
goal_names_extra = {"y_MEAN_ABNORM": "Mean abnormal", "y_NEXT_ABNORM": "Next abnormal", "y_MEAN": "Mean"}

In [ ]:
val_counts["N"].describe()

# Step 0&1 - Extraction and cleaning

## ATC

###  Extraction

In [ ]:
! python3 /home/ivm/valid/scripts/steps/preprocess/step0_extract_ATCs.py \
    --fg_ver R14 \
    --data_path /finngen/library-red/finngen_R14/phenotype_2.0/data/finngen_R14_detailed_longitudinal_2.0.txt.gz \
    --res_dir /home/ivm/valid/data/extra_data/processed_data/step0_extract/ \
    --file_delim '\t' \
    --in_col_names FINNGENID EVENT_AGE APPROX_EVENT_DAY CODE1 \
    --out_col_names FINNGENID EVENT_AGE DATE ATC_CODE ATC_FIVE

###  Process - min PCT

In [ ]:
! python3 /home/ivm/valid/scripts/steps/preprocess/step1_process_ATCs.py \
    --fg_ver R14 \
    --file_path_atcs /home/ivm/valid/data/extra_data/processed_data/step0_extract/ATC_long_R14_2026-04-14.parquet \
    --res_dir /home/ivm/valid/data/extra_data/processed_data/step1_clean/ \
    --min_pct 0.01 \
    --min_age 18 \
    --max_age 70

## ICD

###  Extraction

In [ ]:
! python3 /home/ivm/valid/scripts/steps/preprocess/extract/step0_extract_ICD.py \
    --fg_ver R14 \
    --data_path /finngen/library-red/finngen_R14/phenotype_2.0/data/finngen_R14_detailed_longitudinal_2.0.txt.gz \
    --res_dir /home/ivm/valid/data/extra_data/processed_data/step0_extract/ \
    --file_delim '\t' \
    --in_col_names FINNGENID EVENT_AGE APPROX_EVENT_DAY ICD_CODE CATEGORY ICDVER SOURCE \
    --out_col_names FINNGENID EVENT_AGE DATE ICD_CODE CATEGORY ICD_VERSION SOURCE ICD_THREE

###  Process - min PCT

## Process - min PCT

In [ ]:
! python3 /home/ivm/valid/scripts/steps/preprocess/step1_process_ICDs.py \
    --fg_ver R14 \
    --file_path_icds /home/ivm/valid/data/extra_data/processed_data/step0_extract/ICD10_long_R14_2026-04-13.parquet \
    --res_dir /home/ivm/valid/data/extra_data/processed_data/step1_clean/ \
    --min_pct 0.01 \
    --min_age 18 \
    --max_age 70 \
    --memory_save_mode 1

## Step 0 - Extraction

###  HbA1c

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step0_extract.py \
    --omop=3004410 \
    --res_dir=/home/ivm/valid/data/processed_data/kanta_R14/step0_extract/ \
    --lab_name=hba1c \
    --table_path=/finngen/library-red/finngen_R14/kanta_lab_1.0/data/finngen_R14_kanta_lab_1.0.parquet

###  Fasting glucose

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step0_extract.py \
    --omop=3018251 \
    --res_dir=/home/ivm/valid/data/processed_data/kanta_R14/step0_extract/ \
    --lab_name=fgluc \
    --table_path=/finngen/library-red/finngen_R14/kanta_lab_1.0/data/finngen_R14_kanta_lab_1.0.parquet

###  Glucose

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step0_extract.py \
    --omop=3013826 \
    --res_dir=/home/ivm/valid/data/processed_data/kanta_R14/step0_extract/ \
    --lab_name=gluc \
    --table_path=/finngen/library-red/finngen_R14/kanta_lab_1.0/data/finngen_R14_kanta_lab_1.0.parquet

## Step 1 - Cleaning

###  Glucose

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step1_clean.py \
    --res_dir=/home/ivm/valid/data/processed_data/kanta_R14/step1_clean/ \
    --file_path=/home/ivm/valid/data/processed_data/kanta_R14/step0_extract/gluc_2026-03-23.parquet \
    --lab_name=gluc \
    --fill_missing 0 \
    --ref_min 0.01 \
    --main_unit mmol/l \
    --keep_last_of_day 1 \
    --ref_max 5000 \
    --max_z 50 \
    --plot 1

###  fasting Glucose

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step1_clean.py \
    --res_dir=/home/ivm/valid/data/processed_data/kanta_R14/step1_clean/ \
    --file_path=/home/ivm/valid/data/processed_data/kanta_R14/step0_extract/fgluc_2026-03-23.parquet \
    --lab_name=fgluc \
    --fill_missing 0 \
    --ref_min 0.01 \
    --ref_max 5000 \
    --main_unit mmol/l \
    --keep_last_of_day 1 \
    --max_z 50 \
    --plot 1

###  HbA1c

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step1_clean.py \
    --res_dir=/home/ivm/valid/data/processed_data/kanta_R14/step1_clean/ \
    --file_path=/home/ivm/valid/data/processed_data/kanta_R14/step0_extract/hba1c_2026-03-23.parquet \
    --lab_name=hba1c \
    --fill_missing 0 \
    --ref_min 20 \
    --ref_max 200 \
    --main_unit mmol/mol \
    --abnorm_type strong \
    --keep_last_of_day 1 \
    --max_z 15 \
    --plot 1

## Step 2 - Other diagnosis data

In [ ]:
# **For exclusions and ICD-code or ATC-code based diagnoses**

! python3 /home/ivm/valid/scripts/steps/step2_diags.py \
                --lab_name=hba1c \
                --res_dir=/home/ivm/valid/data/processed_data/kanta_R14/step2_diags/  \
                --diag_regex="(^(E1[0-4]))|(^O24)" --med_regex="^A10" \
                --fg_ver="R14"  \
                --atc_file_path /home/ivm/valid/data/extra_data/processed_data/step0_extract/ATC_long_R14_2026-04-14.parquet \
                --icd_file_path /home/ivm/valid/data/extra_data/processed_data/step0_extract/ICD10_long_R14_2026-04-13.parquet

# Step 2 - Exclusions

In [ ]:
base_file_name = "hba1c_d0_strong_ld_2026-04-20"

diags_data = pl.read_parquet("/home/ivm/valid/data/processed_data/kanta_R14/step2_diags/hba1c_R14_2026-04-22_diags.parquet").with_columns(pl.col.DIAG_DATE.str.to_date())
meds_data = pl.read_parquet("/home/ivm/valid/data/processed_data/kanta_R14/step2_diags/hba1c_R14_2026-04-22_meds.parquet").with_columns(pl.col.MED_DATE.str.to_date())
hba1c_data = pl.read_parquet(base_path+"/step1_clean/"+base_file_name+".parquet")
fgluc_data = pl.read_parquet(base_path+"/step1_clean/fgluc_d0_ld_2026-03-24.parquet")
gluc_data = pl.read_parquet(base_path+"/step1_clean/gluc_d0_ld_2026-03-24.parquet")

fgluc_diag = (fgluc_data
                .filter(pl.col.DATE<base_date,pl.col.VALUE>=7)
                .filter((pl.col.DATE>pl.col.DATE.min()).over("FINNGENID"))
                .filter((pl.col.DATE==pl.col.DATE.min()).over("FINNGENID"))
)
fgluc_val_diag = (fgluc_data
                .filter(pl.col.DATE<val_base_date,pl.col.VALUE>=7)
                .filter((pl.col.DATE>pl.col.DATE.min()).over("FINNGENID"))
                .filter((pl.col.DATE==pl.col.DATE.min()).over("FINNGENID"))
)
fgluc_final_diag = (fgluc_data
                .filter(pl.col.DATE<final_base_date,pl.col.VALUE>=7)
                .filter((pl.col.DATE>pl.col.DATE.min()).over("FINNGENID"))
                .filter((pl.col.DATE==pl.col.DATE.min()).over("FINNGENID"))
)

gluc_diag = (gluc_data
                .filter(pl.col.DATE<base_date,pl.col.VALUE>=11)
                .filter((pl.col.DATE>pl.col.DATE.min()).over("FINNGENID"))
                .filter((pl.col.DATE==pl.col.DATE.min()).over("FINNGENID"))
)
gluc_val_diag = (gluc_data
                .filter(pl.col.DATE<val_base_date,pl.col.VALUE>=11)
                .filter((pl.col.DATE>pl.col.DATE.min()).over("FINNGENID"))
                .filter((pl.col.DATE==pl.col.DATE.min()).over("FINNGENID"))
)
gluc_final_diag = (gluc_data
                .filter(pl.col.DATE<final_base_date,pl.col.VALUE>=11)
                .filter((pl.col.DATE>pl.col.DATE.min()).over("FINNGENID"))
                .filter((pl.col.DATE==pl.col.DATE.min()).over("FINNGENID"))
)

## Numbers

In [ ]:
# Abnormal HbA1c
print("Total historical data")
print_count(hba1c_data.filter(pl.col.DATE<base_date))
print("History of abnormal hba1c")
print_count(hba1c_data
 .filter(pl.col.DATE<base_date)
 .filter(pl.col.ABNORM_CUSTOM==1)
)
print("After removing")
print_count(hba1c_data
 .filter(pl.col.DATE<base_date)
 .filter(~((pl.col.ABNORM_CUSTOM==1).any().over("FINNGENID")))
)
print("Including those without data")
print_count(hba1c_data
 .filter((~((pl.col.DATE<base_date)&(pl.col.ABNORM_CUSTOM==1)).any().over("FINNGENID")))
)

In [ ]:
# Abnormal HbA1c
print("Total historical data")
print_count(hba1c_data.filter(pl.col.DATE<val_base_date))
print("History of abnormal hba1c")
print_count(hba1c_data
 .filter(pl.col.DATE<val_base_date)
 .filter(pl.col.ABNORM_CUSTOM==1)
)
print("After removing")
print_count(hba1c_data
 .filter(pl.col.DATE<val_base_date)
 .filter(~((pl.col.ABNORM_CUSTOM==1).any().over("FINNGENID")))
)
print("Including those without data")
print_count(hba1c_data
 .filter((~((pl.col.DATE<val_base_date)&(pl.col.ABNORM_CUSTOM==1)).any().over("FINNGENID")))
)

In [ ]:
# Diag ICD-10 code
print("##### Removing ICD-10 codes")
print_count(hba1c_data
     .filter((~((pl.col.DATE<base_date)&(pl.col.ABNORM_CUSTOM==1)).any().over("FINNGENID")))
 .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
 .filter(pl.col.DIAG_DATE<base_date)
 .drop("DIAG_DATE", "DIAG").unique()
)
display((hba1c_data
     .filter((~((pl.col.DATE<base_date)&(pl.col.ABNORM_CUSTOM==1)).any().over("FINNGENID")))
 .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
 .filter(pl.col.DIAG_DATE<base_date)
 .with_columns(pl.col.DIAG.str.slice(0,3).alias("DIAG"))
)["DIAG"].value_counts().sort("count", descending=True))

print("After removing")
print_count(hba1c_data
     .filter((~((pl.col.DATE<base_date)&(pl.col.ABNORM_CUSTOM==1)).any().over("FINNGENID")))
 .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
 .filter(~(pl.col.DIAG_DATE<base_date).any().over("FINNGENID"))
 .drop("DIAG_DATE", "DIAG").unique()
)

In [ ]:
# Diag ATC code
print("##### Removing ATC codes")
print_count(hba1c_data
     .filter((~((pl.col.DATE<base_date)&(pl.col.ABNORM_CUSTOM==1)).any().over("FINNGENID")))
 .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
 .filter(~(pl.col.DIAG_DATE<base_date).any().over("FINNGENID"))
 .drop("DIAG_DATE", "DIAG").unique()
 .join(meds_data.select("FINNGENID", "MED_DATE", "MED"), on="FINNGENID", how="left")
 .filter(pl.col.MED_DATE<base_date)
 .drop("MED_DATE", "MED").unique()

)

display((hba1c_data
     .filter((~((pl.col.DATE<base_date)&(pl.col.ABNORM_CUSTOM==1)).any().over("FINNGENID")))
 .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
 .filter(~(pl.col.DIAG_DATE<base_date).any().over("FINNGENID"))
 .drop("DIAG_DATE", "DIAG").unique()
 .join(meds_data.select("FINNGENID", "MED_DATE", "MED"), on="FINNGENID", how="left")
 .filter(pl.col.MED_DATE<base_date)
 .with_columns(pl.col.MED.str.slice(0,5).alias("MED"))

)["MED"].value_counts().sort("count", descending=True))
print("After removing")
print_count(hba1c_data
     .filter((~((pl.col.DATE<base_date)&(pl.col.ABNORM_CUSTOM==1)).any().over("FINNGENID")))
 .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
 .filter(~(pl.col.DIAG_DATE<base_date).any().over("FINNGENID"))
 .drop("DIAG_DATE", "DIAG").unique()
 .join(meds_data.select("FINNGENID", "MED_DATE", "MED"), on="FINNGENID", how="left")
 .filter(~(pl.col.MED_DATE<base_date).any().over("FINNGENID"))
 .drop("MED_DATE", "MED").unique()
)

In [ ]:
print("##### Removing Fasting glucse 2x>=7")
print_count(hba1c_data
     .filter((~((pl.col.DATE<base_date)&(pl.col.ABNORM_CUSTOM==1)).any().over("FINNGENID")))
 .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
 .filter(~(pl.col.DIAG_DATE<base_date).any().over("FINNGENID"))
 .drop("DIAG_DATE", "DIAG").unique()
 .join(meds_data.select("FINNGENID", "MED_DATE", "MED"), on="FINNGENID", how="left")
 .filter(~(pl.col.MED_DATE<base_date).any().over("FINNGENID"))
 .drop("MED_DATE", "MED").unique()
 .filter(pl.col.FINNGENID.is_in(fgluc_diag["FINNGENID"]))
)
print_count(hba1c_data
     .filter((~((pl.col.DATE<base_date)&(pl.col.ABNORM_CUSTOM==1)).any().over("FINNGENID")))
 .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
 .filter(~(pl.col.DIAG_DATE<base_date).any().over("FINNGENID"))
 .drop("DIAG_DATE", "DIAG").unique()
 .join(meds_data.select("FINNGENID", "MED_DATE", "MED"), on="FINNGENID", how="left")
 .filter(~(pl.col.MED_DATE<base_date).any().over("FINNGENID"))
 .drop("MED_DATE", "MED").unique()
 .filter(~pl.col.FINNGENID.is_in(fgluc_diag["FINNGENID"]))
)

In [ ]:
print("##### Removing Glucose 2x>=11")

print_count(hba1c_data
     .filter((~((pl.col.DATE<base_date)&(pl.col.ABNORM_CUSTOM==1)).any().over("FINNGENID")))
 .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
 .filter(~(pl.col.DIAG_DATE<base_date).any().over("FINNGENID"))
 .drop("DIAG_DATE", "DIAG").unique()
 .join(meds_data.select("FINNGENID", "MED_DATE", "MED"), on="FINNGENID", how="left")
 .filter(~(pl.col.MED_DATE<base_date).any().over("FINNGENID"))
 .drop("MED_DATE", "MED").unique()
 .filter(~pl.col.FINNGENID.is_in(fgluc_diag["FINNGENID"]))
 .filter(pl.col.FINNGENID.is_in(gluc_diag["FINNGENID"]))

)
print_count(hba1c_data
     .filter((~((pl.col.DATE<base_date)&(pl.col.ABNORM_CUSTOM==1)).any().over("FINNGENID")))
 .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
 .filter(~(pl.col.DIAG_DATE<base_date).any().over("FINNGENID"))
 .drop("DIAG_DATE", "DIAG").unique()
 .join(meds_data.select("FINNGENID", "MED_DATE", "MED"), on="FINNGENID", how="left")
 .filter(~(pl.col.MED_DATE<base_date).any().over("FINNGENID"))
 .drop("MED_DATE", "MED").unique()
 .filter(~pl.col.FINNGENID.is_in(fgluc_diag["FINNGENID"]))
 .filter(~pl.col.FINNGENID.is_in(gluc_diag["FINNGENID"]))
)

## Filtering

###  Train

In [ ]:
train_data = (hba1c_data
    # this way makes sure that also those without history are there
     .filter((~((pl.col.DATE<base_date)&(pl.col.ABNORM_CUSTOM==1)).any().over("FINNGENID")))
    .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
    .filter(~(pl.col.DIAG_DATE<base_date).any().over("FINNGENID"))
    .drop("DIAG_DATE", "DIAG").unique()
    .join(meds_data.select("FINNGENID", "MED_DATE", "MED"), on="FINNGENID", how="left")
    .filter(~(pl.col.MED_DATE<base_date).any().over("FINNGENID"))
    .drop("MED_DATE", "MED").unique()
    .filter(~pl.col.FINNGENID.is_in(fgluc_diag["FINNGENID"]))
    .filter(~pl.col.FINNGENID.is_in(gluc_diag["FINNGENID"]))
)

train_data.write_parquet(base_path+"/step3_labels/"+base_file_name+"_filtered_"+get_date()+".parquet")

###  Validation

In [ ]:
val_data = (hba1c_data
    # this way makes sure that also those without history are there
    .filter((~((pl.col.DATE<val_base_date)&(pl.col.ABNORM_CUSTOM==1)).any().over("FINNGENID")))
 .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
 .filter(~(pl.col.DIAG_DATE<val_base_date).any().over("FINNGENID"))
 .drop("DIAG_DATE", "DIAG").unique()
 .join(meds_data.select("FINNGENID", "MED_DATE", "MED"), on="FINNGENID", how="left")
 .filter(~(pl.col.MED_DATE<val_base_date).any().over("FINNGENID"))
 .drop("MED_DATE", "MED").unique()
 .filter(~pl.col.FINNGENID.is_in(fgluc_val_diag["FINNGENID"]))
 .filter(~pl.col.FINNGENID.is_in(gluc_val_diag["FINNGENID"]))
)

val_data.write_parquet(base_path+"/step3_labels/"+base_file_name+"_filtered_"+val_name+"_"+get_date()+".parquet")

###  Final prediction

In [ ]:
final_data = (hba1c_data
 .filter((~((pl.col.DATE<final_base_date)&(pl.col.ABNORM_CUSTOM==1)).any().over("FINNGENID")))
 .join(diags_data.select("FINNGENID", "DIAG_DATE", "DIAG"), on="FINNGENID", how="left")
 .filter(~(pl.col.DIAG_DATE<final_base_date).any().over("FINNGENID"))
 .drop("DIAG_DATE", "DIAG").unique()
 .join(meds_data.select("FINNGENID", "MED_DATE", "MED"), on="FINNGENID", how="left")
 .filter(~(pl.col.MED_DATE<final_base_date).any().over("FINNGENID"))
 .drop("MED_DATE", "MED").unique()
 .filter(~pl.col.FINNGENID.is_in(fgluc_final_diag["FINNGENID"]))
 .filter(~pl.col.FINNGENID.is_in(gluc_final_diag["FINNGENID"]))

)

final_data = add_ages(final_data, fg_ver="R13", date=final_base_date)

final_data = final_data.filter(pl.col.EVENT_AGE>=30, pl.col.EVENT_AGE<=70)
final_data = final_data.with_columns(pl.Series("y_MEAN_ABNORM", [0]*final_data.height))
final_data[["FINNGENID", "SEX", "EVENT_AGE", "y_MEAN_ABNORM"]].unique().write_parquet(base_path+"/step3_labels/"+base_file_name+"_filtered_final_"+get_date()+".parquet")

# Step 3 - Labels

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step3_labeling.py \
    --data_path_full "$base_path"/step3_labels/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2".parquet \
    --val_data_path_full "$base_path"/step3_labels/"$lab_name"_"$extra"_"$date_1"_filtered_"$val_name"_"$date_2".parquet \
    --res_dir "$base_path"/step3_labels/ \
    --lab_name "$lab_name" \
    --start_pred_date "$start_pred_date" --end_pred_date "$end_pred_date" \
    --val_start_pred_date "$val_start_pred_date" --val_end_pred_date "$val_end_pred_date" \
    --min_age "$min_age" --max_age "$max_age" \
    --months_buffer "$months_buffer" \
    --abnorm_type "$abnorm_type" \
    --future_val "$future_val" \
    --n_test "$n_test" \
    --version "$version" \
    --test_bbs "AURIA BIOBANK" "TAMPERE BIOBANK"

# Step 4 - Extra Data

## HbA1c sumstats

###  Training

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step4_sumstats.py \
    --res_dir "$base_path"/step4_data/ \
    --file_path "$base_path"/step3_labels/ \
    --file_name_start "$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3" \
    --lab_name "$lab_name" \
    --start_date "$base_date" \
    --mean_impute 0

###  Validation

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step4_sumstats.py \
    --res_dir "$base_path"/step4_data/ \
    --file_path "$base_path"/step3_labels/ \
    --file_name_start "$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3" \
    --lab_name "$lab_name" \
    --start_date "$val_base_date" \
    --mean_impute 0

###  Final prediction

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step4_sumstats.py \
    --res_dir "$base_path"/step4_data/ \
    --file_path "$base_path"/step3_labels/ \
    --file_name_start "$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3" \
    --lab_name "$lab_name" \
    --start_date "$final_base_date" \
    --mean_impute 0

## Fasting glucose sumstats

###  Training

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step4_sumstats.py \
    --res_dir "$base_path"/step4_data/ \
    --file_path_labels "$base_path"/step3_labels/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_labels.parquet \
    --file_path_data "$base_path"/step1_clean/"$lab_name_two"_"$extra_two"_"$date_two".parquet \
    --file_path "$base_path"/step3_labels/ \
    --file_name_start "$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_"$lab_name_two"_"$extra_two"_"$date_two" \
    --lab_name "$lab_name" \
    --start_date "$base_date"

###  Validation

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step4_sumstats.py \
    --res_dir "$base_path"/step4_data/ \
    --file_path_labels "$base_path"/step3_labels/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_labels.parquet \
    --file_path_data "$base_path"/step1_clean/"$lab_name_two"_"$extra_two"_"$date_two".parquet \
    --file_path "$base_path"/step3_labels/ \
    --file_name_start "$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_"$lab_name_two"_"$extra_two"_"$date_two" \
    --lab_name "$lab_name" \
    --start_date "$val_base_date"

###  Final prediction

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step4_sumstats.py \
    --res_dir "$base_path"/step4_data/ \
    --file_path_labels "$base_path"/step3_labels/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_labels.parquet \
    --file_path_data "$base_path"/step1_clean/"$lab_name_two"_"$extra_two"_"$date_two".parquet \
    --file_path "$base_path"/step3_labels/ \
    --file_name_start "$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_"$lab_name_two"_"$extra_two"_"$date_two" \
    --lab_name "$lab_name" \
    --start_date "$final_base_date"

## Other labs

###  Training

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step4_labs.py \
    --res_dir "$base_path"/step4_data/ \
    --file_path_lab /home/ivm/valid/data/extra_data/processed_data/step1_clean/kanta_lab_R14_2026-03-13.parquet \
    --egfr_data_path /home/ivm/valid/data/processed_data/step1_clean/egfr_d0_KDIGO-soft2_ld_2026-03-13.parquet \
    --start_date "$base_date" \
    --fg_ver R14

###  Validation

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step4_labs.py \
    --res_dir "$base_path"/step4_data/ \
    --file_path_lab /home/ivm/valid/data/extra_data/processed_data/step1_clean/kanta_lab_R14_2026-03-13.parquet \
    --egfr_data_path /home/ivm/valid/data/processed_data/step1_clean/egfr_d0_KDIGO-soft2_ld_2026-03-13.parquet \
    --start_date "$val_base_date" \
    --fg_ver R14

###  Final prediction

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step4_labs.py \
    --res_dir "$base_path"/step4_data/ \
    --file_path_lab /home/ivm/valid/data/extra_data/processed_data/step1_clean/kanta_lab_R14_2026-03-13.parquet \
    --egfr_data_path /home/ivm/valid/data/processed_data/step1_clean/egfr_d0_KDIGO-soft2_ld_2026-03-13.parquet \
    --start_date "$final_base_date" \
    --fg_ver R14

## ICDs

###  Training

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step4_atcsicds.py \
            --res_dir "$base_path"/step4_data/ \
            --file_path_preds /home/ivm/valid/data/extra_data/processed_data/step1_clean/icds_R14_min1pct_18t70_2026-04-15.parquet \
            --dir_path_labels "$base_path"/step3_labels/ \
            --file_name_labels_start "$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"\
            --start_date "$base_date" \
            --col_name ICD_THREE \
            --bin_count 1 \
            --start_year 2014 \
            --min_pct 1 \
            --fg_ver R14

###  Validation

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step4_atcsicds.py \
            --res_dir "$base_path"/step4_data/ \
            --file_path_preds /home/ivm/valid/data/extra_data/processed_data/step1_clean/icds_R14_min1pct_18t70_2026-04-15.parquet \
            --dir_path_labels "$base_path"/step3_labels/ \
            --file_name_labels_start "$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"\
            --start_date "$val_base_date" \
            --col_name ICD_THREE \
            --bin_count 1 \
            --start_year 2014 \
            --min_pct 1 \
            --fg_ver R14

###  Final prediction

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step4_atcsicds.py \
            --res_dir "$base_path"/step4_data/ \
            --file_path_preds /home/ivm/valid/data/extra_data/processed_data/step1_clean/icds_R14_min1pct_18t70_2026-04-15.parquet \
            --dir_path_labels "$base_path"/step3_labels/ \
            --file_name_labels_start "$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"\
            --start_date "$final_base_date" \
            --col_name ICD_THREE \
            --bin_count 1 \
            --start_year 2014 \
            --min_pct 1 \
            --fg_ver R14

## ATCs

###  Training

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step4_atcsicds.py \
            --res_dir "$base_path"/step4_data/ \
            --file_path_preds /home/ivm/valid/data/extra_data/processed_data/step1_clean/atcs_R14_min1pct_18t70_2026-04-14.parquet \
            --dir_path_labels "$base_path"/step3_labels/ \
            --file_name_labels_start "$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"\
            --start_date "$base_date" \
            --col_name ATC_FIVE \
            --bin_count 1 \
            --start_year 2014 \
            --min_pct 1 \
            --fg_ver R14

###  Validation

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step4_atcsicds.py \
            --res_dir "$base_path"/step4_data/ \
            --file_path_preds /home/ivm/valid/data/extra_data/processed_data/step1_clean/atcs_R14_min1pct_18t70_2026-04-14.parquet \
            --dir_path_labels "$base_path"/step3_labels/ \
            --file_name_labels_start "$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"\
            --start_date "$val_base_date" \
            --col_name ATC_FIVE \
            --bin_count 1 \
            --start_year 2014 \
            --min_pct 1\
            --fg_ver R14

###  Final prediction

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step4_atcsicds.py \
            --res_dir "$base_path"/step4_data/ \
            --file_path_preds /home/ivm/valid/data/extra_data/processed_data/step1_clean/atcs_R14_min1pct_18t70_2026-04-14.parquet \
            --dir_path_labels "$base_path"/step3_labels/ \
            --file_name_labels_start "$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"\
            --start_date "$final_base_date" \
            --col_name ATC_FIVE \
            --bin_count 1 \
            --start_year 2014 \
            --min_pct 1\
            --fg_ver R14

# Step 5 - XGBoost

## Just age and sex

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step5_fit_XGB.py \
            --lab_name $lab_name \
            --lab_name_two $lab_name_two \
            --lr 0.4 \
            --pred_descriptor 0_base \
            --start_date "$base_date" \
            --val_start_date "$val_base_date" \
            --preds EVENT_AGE SEX \
            --file_path_labels "$base_path"/step3_labels/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_4"_labels.parquet \
            --file_path_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_sumstats_ni_pred2023_"$date_4".parquet \
            --file_path_val_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_sumstats_ni_pred2024_"$date_4".parquet \
            --future_val "$future_val" \
            --res_dir "$base_path"/step5_predict/"$extra_labels"/"$bin_goal"/ \
            --goal "$bin_goal" \
            --metric logloss  \
            --low_lr 0.01 \
            --refit 1 \
            --n_trials 300 \
            --fg_ver R14 \
            --device cuda

## PGS

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step5_fit_XGB.py \
            --lab_name $lab_name \
            --lab_name_two $lab_name_two \
            --lr 0.4 \
            --start_date "$base_date" \
            --pred_descriptor 1_pgs \
            --start_date "$base_date" \
            --val_start_date "$val_base_date" \
            --preds PGS1 PC1 PC2 PC3 PC4 PC5 EVENT_AGE SEX \
            --file_path_labels "$base_path"/step3_labels/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_labels.parquet \
            --file_path_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_sumstats_ni_pred2023_"$date_4".parquet \
            --file_path_val_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_sumstats_ni_pred2024_"$date_4".parquet \
            --file_path_pgs1 /home/ivm/valid/data/extra_data/pgs/2026-02/HbA1c_PGS002697.txt.sscore \
            --future_val "$future_val" \
            --res_dir "$base_path"/step5_predict/"$extra_labels"/"$bin_goal"/ \
            --goal "$bin_goal" \
            --metric logloss  \
            --low_lr 0.01 \
            --refit 1 \
            --n_trials 300 \
            --fg_ver R14 \
            --device cuda

## Last value

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step5_fit_XGB.py \
            --lab_name $lab_name \
            --lab_name_two $lab_name_two \
            --lr 0.4 \
            --pred_descriptor 1_lastval \
            --start_date "$base_date" \
            --val_start_date "$val_base_date" \
            --preds IDX_QUANT_100 LAST_VAL_DIFF EVENT_AGE SEX \
            --file_path_labels "$base_path"/step3_labels/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_labels.parquet \
            --file_path_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_sumstats_ni_pred2023_"$date_4".parquet \
            --file_path_val_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_sumstats_ni_pred2024_"$date_4".parquet \
            --future_val "$future_val" \
            --res_dir "$base_path"/step5_predict/"$extra_labels"/"$bin_goal"/ \
            --goal "$bin_goal" \
            --metric logloss  \
            --low_lr 0.01 \
            --refit 1 \
            --n_trials 300 \
            --fg_ver R14 \
            --device cuda

## Last value + PGS

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step5_fit_XGB.py \
            --lab_name $lab_name \
            --lab_name_two $lab_name_two \
            --lr 0.4 \
            --start_date "$base_date" \
            --val_start_date "$val_base_date" \
            --pred_descriptor 91_lastval_pgs \
            --preds PGS1 PC1 PC2 PC3 PC4 PC5 IDX_QUANT_100 LAST_VAL_DIFF EVENT_AGE SEX \
            --file_path_labels "$base_path"/step3_labels/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_labels.parquet \
            --file_path_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_sumstats_ni_pred2023_"$date_4".parquet \
            --file_path_val_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_sumstats_ni_pred2024_"$date_4".parquet \
            --file_path_pgs1 /home/ivm/valid/data/extra_data/pgs/2026-02/HbA1c_PGS002697.txt.sscore \
            --future_val "$future_val" \
            --res_dir "$base_path"/step5_predict/"$extra_labels"/"$bin_goal"/ \
            --goal "$bin_goal" \
            --metric logloss  \
            --low_lr 0.01 \
            --refit 1 \
            --n_trials 300 \
            --fg_ver R14 \
            --device cuda

## Sumstats +  PGS

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step5_fit_XGB.py \
            --lab_name $lab_name \
            --lab_name_two $lab_name_two \
            --lr 0.4 \
            --start_date "$base_date" \
            --val_start_date "$val_base_date" \
            --pred_descriptor 92_sumstats_pgs \
            --preds PGS1 PC1 PC2 PC3 PC4 PC5 SUMSTATS LAST_VAL_DIFF EVENT_AGE SEX \
            --file_path_labels "$base_path"/step3_labels/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_labels.parquet \
            --file_path_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_sumstats_ni_pred2023_"$date_4".parquet \
            --file_path_val_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_sumstats_ni_pred2024_"$date_4".parquet \
            --file_path_pgs1 /home/ivm/valid/data/extra_data/pgs/2026-02/HbA1c_PGS002697.txt.sscore \
            --future_val "$future_val" \
            --res_dir "$base_path"/step5_predict/"$extra_labels"/"$bin_goal"/ \
            --goal "$bin_goal" \
            --metric logloss  \
            --low_lr 0.01 \
            --refit 1 \
            --n_trials 300 \
            --fg_ver R14 \
            --device cuda

## Sumstats

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step5_fit_XGB.py \
            --lab_name $lab_name \
            --lab_name_two $lab_name_two \
            --lr 0.4 \
            --start_date "$base_date" \
            --val_start_date "$val_base_date" \
            --pred_descriptor 2_sumstats \
            --preds SUMSTATS LAST_VAL_DIFF EVENT_AGE SEX \
            --file_path_labels "$base_path"/step3_labels/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_labels.parquet \
            --file_path_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_sumstats_ni_pred2023_"$date_4".parquet \
            --file_path_val_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_sumstats_ni_pred2024_"$date_4".parquet \
            --future_val "$future_val" \
            --res_dir "$base_path"/step5_predict/"$extra_labels"/"$bin_goal"/ \
            --goal "$bin_goal" \
            --metric logloss  \
            --low_lr 0.01 \
            --refit 1 \
            --n_trials 300 \
            --fg_ver R14 \
            --device cuda

## Other Sumstats

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step5_fit_XGB.py \
            --lab_name $lab_name \
            --lab_name_two $lab_name_two \
            --lr 0.4 \
            --start_date "$base_date" \
            --val_start_date "$val_base_date" \
            --pred_descriptor 3_sumstats_o \
            --preds SECOND_SUMSTATS EVENT_AGE SEX \
            --file_path_labels "$base_path"/step3_labels/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_labels.parquet \
            --file_path_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_sumstats_ni_pred2023_"$date_4".parquet \
            --file_path_val_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_sumstats_ni_pred2024_"$date_4".parquet \
            --file_path_second_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_"$lab_name_two"_"$extra_two"_"$date_two"_sumstats_pred2023_"$date_4".parquet \
            --file_path_val_second_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_"$lab_name_two"_"$extra_two"_"$date_two"_sumstats_pred2024_"$date_4".parquet \
            --res_dir "$base_path"/step5_predict/"$extra_labels"/"$bin_goal"/ \
            --future_val "$future_val" \
            --goal "$bin_goal" \
            --metric logloss  \
            --low_lr 0.01 \
            --refit 1 \
            --n_trials 300 \
            --fg_ver R14 \
            --device cuda

## Other labs

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step5_fit_XGB.py \
            --lab_name $lab_name \
            --lab_name_two $lab_name_two \
            --lr 0.4 \
            --start_date "$base_date" \
            --val_start_date "$val_base_date" \
            --pred_descriptor 3_otherlabs \
            --preds S_MEAN LAB_MAT_MEAN EVENT_AGE SEX \
            --file_path_labels "$base_path"/step3_labels/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_labels.parquet \
            --file_path_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_sumstats_ni_pred2023_"$date_4".parquet \
            --file_path_val_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_sumstats_ni_pred2024_"$date_4".parquet \
            --file_path_second_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_"$lab_name_two"_"$extra_two"_"$date_two"_sumstats_pred2023_"$date_4".parquet \
            --file_path_val_second_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_"$lab_name_two"_"$extra_two"_"$date_two"_sumstats_pred2024_"$date_4".parquet \
            --file_path_labs "$base_path"/step4_data/labs_pred2023_"$date_5".parquet \
            --file_path_val_labs "$base_path"/step4_data/labs_pred2024_"$date_5".parquet \
            --res_dir "$base_path"/step5_predict/"$extra_labels"/"$bin_goal"/ \
            --future_val "$future_val" \
            --goal "$bin_goal" \
            --metric logloss  \
            --low_lr 0.01 \
            --refit 1 \
            --n_trials 300 \
            --fg_ver R14 \
            --device cuda

## Registry data

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step5_fit_XGB.py \
            --lab_name $lab_name \
            --lab_name_two $lab_name_two \
            --lr 0.4 \
            --start_date "$base_date" \
            --val_start_date "$val_base_date" \
            --pred_descriptor 3_registry \
            --preds ICD_MAT ATC_MAT EVENT_AGE SEX \
            --file_path_labels "$base_path"/step3_labels/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_labels.parquet \
            --file_path_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_sumstats_ni_pred2023_"$date_4".parquet \
            --file_path_val_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_sumstats_ni_pred2024_"$date_4".parquet \
            --file_path_icds "$base_path"/step4_data/icds_1pct_bin_start_2014_pred2023_"$date_5".parquet \
            --file_path_val_icds "$base_path"/step4_data/icds_1pct_bin_start_2014_pred2024_"$date_5".parquet \
            --file_path_atcs "$base_path"/step4_data/atcs_1pct_bin_start_2014_pred2023_"$date_5".parquet \
            --file_path_val_atcs "$base_path"/step4_data/atcs_1pct_bin_start_2014_pred2024_"$date_5".parquet \
            --res_dir "$base_path"/step5_predict/"$extra_labels"/"$bin_goal"/ \
            --future_val "$future_val" \
            --goal "$bin_goal" \
            --metric logloss  \
            --low_lr 0.01 \
            --refit 1 \
            --n_trials 300 \
            --fg_ver R14 \
            --device cuda

## All

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step5_fit_XGB.py \
            --lab_name $lab_name \
            --lab_name_two $lab_name_two \
            --lr 0.4 \
            --start_date "$base_date" \
            --pred_descriptor 4_all \
            --start_date "$base_date" \
            --val_start_date "$val_base_date" \
            --preds SUMSTATS S_MEAN LAST_VAL_DIFF EDU BMI LAB_MAT_MEAN ICD_MAT ATC_MAT EVENT_AGE SEX \
            --file_path_labels "$base_path"/step3_labels/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_labels.parquet \
            --file_path_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_sumstats_ni_pred2023_"$date_4".parquet \
            --file_path_val_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_sumstats_ni_pred2024_"$date_4".parquet \
            --file_path_second_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_"$lab_name_two"_"$extra_two"_"$date_two"_sumstats_pred2023_"$date_4".parquet \
            --file_path_val_second_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_"$lab_name_two"_"$extra_two"_"$date_two"_sumstats_pred2024_"$date_4".parquet \
            --file_path_labs "$base_path"/step4_data/labs_pred2023_"$date_5".parquet \
            --file_path_val_labs "$base_path"/step4_data/labs_pred2024_"$date_5".parquet \
            --file_path_icds "$base_path"/step4_data/icds_1pct_bin_start_2014_pred2023_"$date_5".parquet \
            --file_path_val_icds "$base_path"/step4_data/icds_1pct_bin_start_2014_pred2024_"$date_5".parquet \
            --file_path_atcs "$base_path"/step4_data/atcs_1pct_bin_start_2014_pred2023_"$date_5".parquet \
            --file_path_val_atcs "$base_path"/step4_data/atcs_1pct_bin_start_2014_pred2024_"$date_5".parquet \
            --res_dir "$base_path"/step5_predict/"$extra_labels"/"$bin_goal"/ \
            --future_val "$future_val" \
            --goal "$bin_goal" \
            --metric logloss  \
            --low_lr 0.01 \
            --refit 1 \
            --n_trials 300 \
            --fg_ver R14 \
            --device cuda

## All+PGS

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step5_fit_XGB.py \
            --lab_name $lab_name \
            --lab_name_two $lab_name_two \
            --lr 0.4 \
            --start_date "$base_date" \
            --val_start_date "$val_base_date" \
            --pred_descriptor 5_all_pgs \
            --preds PGS1 PC1 PC2 PC3 PC4 PC5 SUMSTATS S_MEAN LAST_VAL_DIFF EDU BMI LAB_MAT_MEAN ICD_MAT ATC_MAT EVENT_AGE SEX \
            --file_path_labels "$base_path"/step3_labels/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_labels.parquet \
            --file_path_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_sumstats_ni_pred2023_"$date_4".parquet \
            --file_path_val_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_sumstats_ni_pred2024_"$date_4".parquet \
            --file_path_second_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_"$lab_name_two"_"$extra_two"_"$date_two"_sumstats_pred2023_"$date_4".parquet \
            --file_path_val_second_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_"$lab_name_two"_"$extra_two"_"$date_two"_sumstats_pred2024_"$date_4".parquet \
            --file_path_labs "$base_path"/step4_data/labs_pred2023_"$date_5".parquet \
            --file_path_val_labs "$base_path"/step4_data/labs_pred2024_"$date_5".parquet \
            --file_path_icds "$base_path"/step4_data/icds_1pct_bin_start_2014_pred2023_"$date_5".parquet \
            --file_path_val_icds "$base_path"/step4_data/icds_1pct_bin_start_2014_pred2024_"$date_5".parquet \
            --file_path_atcs "$base_path"/step4_data/atcs_1pct_bin_start_2014_pred2023_"$date_5".parquet \
            --file_path_val_atcs "$base_path"/step4_data/atcs_1pct_bin_start_2014_pred2024_"$date_5".parquet \
            --file_path_pgs1 /home/ivm/valid/data/extra_data/pgs/2026-02/HbA1c_PGS002697.txt.sscore \
            --res_dir "$base_path"/step5_predict/"$extra_labels"/"$bin_goal"/ \
            --future_val "$future_val" \
            --goal "$bin_goal" \
            --metric logloss  \
            --low_lr 0.01 \
            --refit 1 \
            --n_trials 300 \
            --fg_ver R14 \
            --device cuda

## All with only mean + PGS

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step5_fit_XGB.py \
            --lab_name $lab_name \
            --lab_name_two $lab_name_two \
            --lr 0.4 \
            --start_date "$base_date" \
            --val_start_date "$val_base_date" \
            --pred_descriptor 95_all_pgs \
            --start_date "$base_date" \
            --preds PGS1 PC1 PC2 PC3 PC4 PC5 MEAN S_MEAN LAST_VAL_DIFF EDU BMI LAB_MAT_MEAN ICD_MAT ATC_MAT EVENT_AGE SEX \
            --file_path_labels "$base_path"/step3_labels/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_labels.parquet \
            --file_path_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_sumstats_ni_pred2023_"$date_4".parquet \
            --file_path_val_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_sumstats_ni_pred2024_"$date_4".parquet \
            --file_path_second_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_"$lab_name_two"_"$extra_two"_"$date_two"_sumstats_pred2023_"$date_4".parquet \
            --file_path_val_second_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_"$lab_name_two"_"$extra_two"_"$date_two"_sumstats_pred2024_"$date_4".parquet \
            --file_path_labs "$base_path"/step4_data/labs_pred2023_"$date_5".parquet \
            --file_path_val_labs "$base_path"/step4_data/labs_pred2024_"$date_5".parquet \
            --file_path_icds "$base_path"/step4_data/icds_1pct_bin_start_2014_pred2023_"$date_5".parquet \
            --file_path_val_icds "$base_path"/step4_data/icds_1pct_bin_start_2014_pred2024_"$date_5".parquet \
            --file_path_atcs "$base_path"/step4_data/atcs_1pct_bin_start_2014_pred2023_"$date_5".parquet \
            --file_path_val_atcs "$base_path"/step4_data/atcs_1pct_bin_start_2014_pred2024_"$date_5".parquet \
            --file_path_pgs1 /home/ivm/valid/data/extra_data/pgs/2026-02/HbA1c_PGS002697.txt.sscore \
            --res_dir "$base_path"/step5_predict/"$extra_labels"/"$bin_goal"/ \
            --future_val "$future_val" \
            --goal "$bin_goal" \
            --metric logloss  \
            --low_lr 0.01 \
            --refit 1 \
            --n_trials 300 \
            --fg_ver R14 \
            --device cuda

# Step 6 - Final fit

In [ ]:
! python3 /home/ivm/valid/scripts/steps/step5_fit_XGB.py \
            --lab_name $lab_name \
            --lab_name_two $lab_name_two \
            --lr 0.4 \
            --start_date "$base_date" \
            --val_start_date "$val_base_date" \
            --pred_descriptor final_5_all_pgs \
            --start_date "$base_date" \
            --preds PGS1 PC1 PC2 PC3 PC4 PC5 SUMSTATS S_MEAN LAST_VAL_DIFF EDU BMI LAB_MAT_MEAN ICD_MAT ATC_MAT EVENT_AGE SEX \
            --file_path_labels "$base_path"/step3_labels/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_labels.parquet \
            --file_path_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_sumstats_ni_pred2024_"$date_4".parquet \
            --file_path_val_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_sumstats_ni_pred2026_"$date_4".parquet \
            --file_path_second_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_"$lab_name_two"_"$extra_two"_"$date_two"_sumstats_pred2024_"$date_4".parquet \
            --file_path_val_second_sumstats "$base_path"/step4_data/"$lab_name"_"$extra"_"$date_1"_filtered_"$date_2"_"$extra_labels"_"$date_3"_"$lab_name_two"_"$extra_two"_"$date_two"_sumstats_pred2026_"$date_4".parquet \
            --file_path_labs "$base_path"/step4_data/labs_pred2024_"$date_5".parquet \
            --file_path_val_labs "$base_path"/step4_data/labs_pred2026_"$date_5".parquet \
            --file_path_icds "$base_path"/step4_data/icds_1pct_bin_start_2014_pred2024_"$date_5".parquet \
            --file_path_val_icds "$base_path"/step4_data/icds_1pct_bin_start_2014_pred2026_"$date_5".parquet \
            --file_path_atcs "$base_path"/step4_data/atcs_1pct_bin_start_2014_pred2024_"$date_5".parquet \
            --file_path_val_atcs "$base_path"/step4_data/atcs_1pct_bin_start_2014_pred2026_"$date_5".parquet \
            --file_path_pgs1 /home/ivm/valid/data/extra_data/pgs/2026-02/HbA1c_PGS002697.txt.sscore \
            --res_dir "$base_path"/step5_predict/"$extra_labels"/"$bin_goal"/ \
            --future_val "$future_val" \
            --goal "$bin_goal" \
            --metric logloss  \
            --low_lr 0.01 \
            --refit 1 \
            --n_trials 300 \
            --fg_ver R14 \
            --device cuda \
            --model_fit_date 2026-04-24 \
            --final_indvs_path "$base_path"/step3_labels/hba1c_d0_strong_ld_2026-04-20_filtered_final_2026-04-24.parquet \
            --final_fit 1

# Step 7 - Results compilation

In [ ]:
from final_comp_utils import final_diff_eval, final_aucs

In [ ]:
no_dups_combos = [
    ("xgb_logloss_0_base", "xgb_logloss_1_lastval"),
    ("xgb_logloss_0_base", "xgb_logloss_1_pgs"),
    ("xgb_logloss_0_base", "xgb_logloss_2_sumstats"),
    ("xgb_logloss_0_base", "xgb_logloss_3_registry"),
    ("xgb_logloss_0_base", "xgb_logloss_3_otherlabs"),
    ("xgb_logloss_0_base", "xgb_logloss_3_sumstats_o"),
    ("xgb_logloss_0_base", "xgb_logloss_4_all"),
    ("xgb_logloss_0_base", "xgb_logloss_5_all_pgs"),
    ("xgb_logloss_0_base", "xgb_logloss_91_lastval_pgs"),
    ("xgb_logloss_0_base", "xgb_logloss_92_sumstats_pgs"),
    ("xgb_logloss_0_base", "xgb_logloss_95_all_pgs"),

    ("xgb_logloss_1_lastval", "xgb_logloss_1_pgs"),
    ("xgb_logloss_1_lastval", "xgb_logloss_2_sumstats"),
    ("xgb_logloss_1_lastval", "xgb_logloss_3_registry"),
    ("xgb_logloss_1_lastval", "xgb_logloss_3_sumstats_o"),
    ("xgb_logloss_1_lastval", "xgb_logloss_3_otherlabs"),
    ("xgb_logloss_1_lastval", "xgb_logloss_4_all"),
    ("xgb_logloss_1_lastval", "xgb_logloss_5_all_pgs"),
    ("xgb_logloss_1_lastval", "xgb_logloss_91_lastval_pgs"),
    
    ("xgb_logloss_2_sumstats", "xgb_logloss_3_sumstats_o"),
    ("xgb_logloss_2_sumstats", "xgb_logloss_3_registry"),
    ("xgb_logloss_2_sumstats", "xgb_logloss_3_otherlabs"),
    ("xgb_logloss_2_sumstats", "xgb_logloss_4_all"),
    ("xgb_logloss_2_sumstats", "xgb_logloss_5_all_pgs"),
    ("xgb_logloss_2_sumstats", "xgb_logloss_92_sumstats_pgs"),

    ("xgb_logloss_3_registry", "xgb_logloss_1_pgs"),
    ("xgb_logloss_3_registry", "xgb_logloss_3_otherlabs"),
    ("xgb_logloss_3_registry", "xgb_logloss_3_sumstats_o"),
    ("xgb_logloss_3_registry", "xgb_logloss_4_all"),
    ("xgb_logloss_3_registry", "xgb_logloss_5_all_pgs"),

    ("xgb_logloss_3_otherlabs", "xgb_logloss_1_pgs"),
    ("xgb_logloss_3_otherlabs", "xgb_logloss_4_all"),
    ("xgb_logloss_3_otherlabs", "xgb_logloss_5_all_pgs"),
    ("xgb_logloss_3_otherlabs", "xgb_logloss_3_sumstats_o"),

    ("xgb_logloss_4_all", "xgb_logloss_5_all_pgs"),
    ("xgb_logloss_5_all_pgs", "xgb_logloss_95_all_pgs"),

]

## Differences

In [ ]:
results = final_diff_eval(lab_name,
                          no_dups_combos,
                          base_path,
                          file_descr,
                          set_names,
                          filters,
                          n_boots=100)
make_dir(base_path+"/model_evals/"+lab_name+"/")
results.write_csv(base_path+"/model_evals/"+lab_name+"_"+file_descr+"_diffs_"+get_date()+".csv")

## AUCs

In [ ]:
results = final_aucs(lab_name=lab_name,
                     base_path=base_path,
                     labels_path="/home/ivm/valid/data/processed_data/kanta_R14/step3_labels/hba1c_d0_strong_ld_2026-04-20_filtered_2026-04-24_v10_2023val2024_w3_2026-04-24_labels.parquet",
                     no_dups_combos=no_dups_combos,
                     file_descr=file_descr,
                     goal_names=goal_names,
                     filters=filters,
                     set_names=set_names,
                     n_boots=100,
                     train_goals=["y_MEAN_ABNORM"],
                     eval_goals=["y_MEAN_ABNORM"],
                     eval_sets=[0,1,2])

make_dir(base_path+"/model_evals/"+lab_name+"/")
results.filter(pl.col.N_CASE>=5).write_csv(base_path+"/model_evals/"+lab_name+"_"+file_descr+"_aucs_"+get_date()+".csv")